
# 1.LOAD ARTIFAK MODEL

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Sesuaikan folder tempat menyimpan hasil training
artifact_dir = Path("artifacts/post_award_anomaly")

print("Memuat artifak model...")
try:
    model = joblib.load(artifact_dir / "isolation_forest.joblib")
    preprocessor = joblib.load(artifact_dir / "preprocessor.joblib")
    
    # Otomatis mengambil nama fitur dari preprocessor
    feature_names_clean = preprocessor.get_feature_names_out()
    
    # Inisialisasi SHAP Explainer untuk mendapatkan alasan prediksi
    explainer_shap = shap.TreeExplainer(model)
    print("Artifak dan nama fitur berhasil dimuat.")
except Exception as e:
    print(f"Gagal memuat artifak: {e}")



# 2. KONFIGURASI NLP ENGINE 

In [ ]:
DAFTAR_HARI = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
KAMUS_INDO = {
    "tender_value": "Nilai tender",
    "award_id": "ID Kontrak",
    "buyer_name": "Nama Instansi",
    "buyer_id": "ID Instansi",
    "award_value": "Nilai kontrak akhir"
}

def format_rupiah(angka):
    """Format angka ke Rupiah dengan titik pemisah ribuan."""
    if pd.isna(angka) or angka is None: return "N/A"
    return f"Rp {angka:,.0f}".replace(',', '.')

def aggregate_shap_to_raw(row_shap, feature_names_list):
    """Menggabungkan kontribusi SHAP ke fitur asli."""
    aggregated = {}
    for feat, val in zip(feature_names_list, row_shap):
        raw = "mainprocurementcategory" if feat.startswith("cat_") else feat
        aggregated[raw] = aggregated.get(raw, 0.0) + val
    return aggregated

# --- Kumpulan Handler untuk Penjelasan Alami ---
def handle_log(feat, num_val, is_aneh):
    try:
        # Mengembalikan nilai Log ke nominal Rupiah asli
        asli_rupiah = np.exp(num_val) if num_val < 100 else num_val
        val_str = format_rupiah(asli_rupiah)
    except:
        val_str = str(round(num_val, 2))

    kamus_konsep = {
        "log_tender_value": "Nilai awal tender",
        "log_award_value": "Nilai kontrak akhir",
        "log_tender_minvalue": "Batas minimum tender",
        "log_value_gap": "Selisih penghematan"
    }
    nama_konsep = kamus_konsep.get(feat, feat.replace("log_", "").replace("_", " ").title())

    if is_aneh:
        return f"Besaran {nama_konsep} (estimasi {val_str}) tercatat tidak wajar dan berisiko bagi anggaran."
    return f"Besaran {nama_konsep} (estimasi {val_str}) terpantau wajar sesuai standar."

def handle_days_to_award(num_val, is_aneh, **kwargs):
    if is_aneh: return f"Proses lelang selesai dalam {int(num_val)} hari, durasi ini tergolong tidak lazim (terlalu cepat/lama)."
    return f"Proses lelang memakan waktu {int(num_val)} hari, merupakan durasi wajar."

def handle_budget_ratio(num_val, is_aneh, **kwargs):
    persen = num_val * 100
    if is_aneh: return f"Tingkat serapan anggaran mencapai {persen:.1f}%, angka ini mencurigakan (indikasi inefisiensi)."
    return f"Tingkat serapan anggaran sebesar {persen:.1f}% dinilai masih dalam batas aman."

def handle_value_gap(num_val, is_aneh, **kwargs):
    if is_aneh: return f"Terdapat selisih tidak proporsional antara nilai tender dan kontrak sebesar {format_rupiah(num_val)}."
    return f"Penghematan anggaran (selisih tender dan kontrak) sebesar {format_rupiah(num_val)} terdeteksi normal."

def handle_award_value_per_day(num_val, is_aneh, **kwargs):
    if is_aneh: return f"Rata-rata pengeluaran harian mencapai {format_rupiah(num_val)}, nominal ini tidak proporsional dengan waktu proyek."
    return f"Estimasi pengeluaran harian sebesar {format_rupiah(num_val)} masih tergolong masuk akal."

def handle_award_title_word_count(num_val, is_aneh, **kwargs):
    if is_aneh: return f"Judul dokumen kontrak menggunakan {int(num_val)} kata, format ini melenceng dari standar administrasi."
    return f"Judul dokumen kontrak terdiri dari {int(num_val)} kata, format penamaan dokumen tergolong standar."

def handle_award_weekday(num_val, is_aneh, **kwargs):
    idx_hari = int(num_val) if 0 <= int(num_val) <= 6 else 0
    nama_hari = DAFTAR_HARI[idx_hari]
    if is_aneh:
        if idx_hari >= 5: return f"Penetapan pemenang dilakukan pada hari libur ({nama_hari}), indikasi ketidakwajaran proses."
        return f"Terdeteksi anomali waktu pada sistem saat penetapan di hari {nama_hari}."
    return f"Penetapan pemenang dilakukan pada hari kerja ({nama_hari}), sesuai jam operasional."

def handle_fallback(feat, num_val, raw_val, is_aneh):
    nama_fitur_bersih = KAMUS_INDO.get(feat, feat.replace("_", " ").title())
    if num_val is not None:
        val_str = format_rupiah(num_val) if num_val > 1000 else str(round(num_val, 2))
    else:
        val_str = str(raw_val)
    if is_aneh: return f"Terdapat deviasi pada metrik '{nama_fitur_bersih}' (nilai: {val_str}) yang butuh tinjauan."
    return f"Metrik '{nama_fitur_bersih}' (nilai: {val_str}) terpantau wajar."

# Dispatcher Mapping untuk kecepatan eksekusi
FEATURE_DISPATCH = {
    "days_to_award": handle_days_to_award, 
    "budget_utilization_ratio": handle_budget_ratio,
    "value_gap": handle_value_gap, 
    "award_value_per_day": handle_award_value_per_day,
    "award_title_word_count": handle_award_title_word_count, 
    "award_weekday": handle_award_weekday
}

def generate_natural_reason(feat, raw_val, shap_val):
    try: num_val = float(raw_val)
    except: num_val = None
    is_aneh = shap_val < 0 
    if feat.startswith("log_") and num_val is not None: return handle_log(feat, num_val, is_aneh)
    handler = FEATURE_DISPATCH.get(feat)
    if handler: return handler(num_val=num_val, raw_val=raw_val, is_aneh=is_aneh)
    return handle_fallback(feat, num_val, raw_val, is_aneh)

# 3. FUNGSI UTAMA INFERENSI

In [8]:
def predict_with_natural_explanation(input_df):
    """Melakukan prediksi dan menghasilkan narasi penjelasan otomatis."""
    # 1. Transformasi & Prediksi
    X_transformed = preprocessor.transform(input_df)
    scores = model.decision_function(X_transformed)
    preds = model.predict(X_transformed)
    
    # 2. Kalkulasi kontribusi fitur (SHAP)
    shap_vals = explainer_shap.shap_values(X_transformed)
    
    results = []
    for i in range(len(input_df)):
        is_anomaly = preds[i] == -1
        label = "ANOMALI" if is_anomaly else "NORMAL"
        
        # Agregasi SHAP ke fitur asli
        agg = aggregate_shap_to_raw(shap_vals[i], feature_names_clean)
        
        # Ambil Top-3 pendorong keputusan
        if is_anomaly:
            sorted_feats = sorted(agg.items(), key=lambda x: x[1])[:3]
        else:
            sorted_feats = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:3]
            
        reasons = []
        for idx, (feat, s_val) in enumerate(sorted_feats, start=1):
            raw_v = input_df.iloc[i].get(feat, "N/A")
            reasons.append(f"Alasan {idx}: {generate_natural_reason(feat, raw_v, s_val)}")
            
        intro = "Sistem mendeteksi dokumen ini MENCURIGAKAN:" if is_anomaly else "Status transaksi dinilai WAJAR DAN AMAN:"
        explanation = f"[{label}] {intro}\n" + "\n".join(reasons)
        
        results.append({
            "anomaly_score": scores[i],
            "prediction_label": label.lower(),
            "human_readable_explanation": explanation
        })
        
    return pd.DataFrame(results)


print("\n--- Memulai Proses Deteksi ---")


--- Memulai Proses Deteksi ---


# 4. EKSEKUSI PADA DATA DEMO

In [ ]:
# Pastikan 'demo_df' telah melewati fungsi rekayasa fitur (feature engineering)
try:
    hasil = predict_with_natural_explanation(demo_df)
    final_report = pd.concat([demo_df.reset_index(drop=True), hasil], axis=1)
    
    # Menampilkan hasil
    pd.set_option('display.max_colwidth', None)
    display(final_report[['prediction_label', 'anomaly_score', 'human_readable_explanation']].head())

except NameError:
    print("Error: Variabel 'demo_df' tidak ditemukan. Pastikan data input sudah siap di cell sebelumnya.")